# In-Class Activity: Prompt Engineering for Construction Documents and Drawings
In this activity, we will use generative AI (GPT/HokieAI) to classify construction incident reports, review submittals against a specification, answer questions from a contract, and read construction drawings, and benchmark how the wording of a prompt changes the accuracy of its outputs.

In [ ]:
#@title ▶ Step 0 · Run me first (1 minute) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line.
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp7_prompt_engineering", "aec_prompt"
FOLDERS = ["mp7_prompt_engineering"]          # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_prompt import lab
lab.setup()


## Part 1 · Classifying Incident Reports
OSHA records every severe work-related injury reported by employers, with a short narrative and a code for the event that caused it. Here, you will use HokieAI to classify 30 construction incident reports into OSHA's construction Focus Four hazards (fall, struck-by, caught-in/between, electrocution) or other, and compare its output with OSHA's own classification.

In every HokieAI step, the notebook gives you a prompt; you send it to HokieAI (https://hokie.ai.vt.edu/), paste HokieAI's output back, and the notebook scores it against the ground truth (the correct output, known for every example). Each prompt version changes one thing, and every scored output is added to the comparison step that follows.

In [ ]:
#@title ▶ Step 1a · Browse the Reports { display-mode: "form" }
#@markdown Run the cell to browse the incident reports HokieAI will classify. OSHA's classification of each report is the ground truth; it stays hidden until you score an output.
rows = 10 #@param [5, 10, 30] {type:"raw"}
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
summary_text = """**{n}** construction incident reports from OSHA's Severe Injury Reports ({years}). OSHA's classification of each report, the ground truth, is hidden."""
lab.show_reports(rows, summary_text=summary_text)


In [ ]:
#@title ▶ Step 1b · Classify with HokieAI { display-mode: "form" }
#@markdown Choose a prompt version, run the cell, and follow the instructions. Score every version, and run one version twice in new chats to check its consistency.
#@markdown v1 · task only: the categories and the output format.
#@markdown v2 · role and definitions: adds a role and OSHA's definition of each category.
#@markdown v3 · definitions and worked examples: adds five reports with OSHA's classification (few-shot prompting).
#@markdown my own prompt: starts from v3; edit the prompt in the prompt box before copying it.
#@markdown If HokieAI declines a prompt or its output is blocked, record which version: that is a result too.
prompt_version = "v1 · task only" #@param ["v1 · task only", "v2 · role and definitions", "v3 · definitions and worked examples", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """Classify each construction incident report below into one category: Fall, Struck-by, Caught-in/between, Electrocution, or Other.
Reply with one line per report and nothing else, in this format:
N01, Fall
N02, Other

Incident reports:
{reports}"""
v2_prompt_text = """You are a construction safety analyst coding OSHA severe-injury reports.
Definitions (OSHA construction Focus Four, as coded in OSHA's injury records):
- Fall: the worker fell to a lower level (from a ladder, roof, scaffold, through an opening), fell on the same level, or jumped down.
- Struck-by: the worker was hit by a moving object or equipment: a falling, flying, swinging, or rolling object, a vehicle, or a tool or object that slipped or swung while the worker was holding it (for example, a saw kickback or a slipping knife).
- Caught-in/between: the worker or a body part was caught in, squeezed, compressed, pinched, or crushed between objects or in running machinery, including trench, structure, or material collapses.
- Electrocution: the worker made contact with electrical energy (power lines, energized wiring or equipment).
- Other: anything else (heat, chemicals, fire or explosion, overexertion, struck against a stationary object, slips or trips without a fall).

Classify each construction incident report below into one category: Fall, Struck-by, Caught-in/between, Electrocution, or Other.
Reply with one line per report and nothing else, in this format:
N01, Fall
N02, Other

Incident reports:
{reports}"""
v3_prompt_text = """You are a construction safety analyst coding OSHA severe-injury reports.
Definitions (OSHA construction Focus Four, as coded in OSHA's injury records):
- Fall: the worker fell to a lower level (from a ladder, roof, scaffold, through an opening), fell on the same level, or jumped down.
- Struck-by: the worker was hit by a moving object or equipment: a falling, flying, swinging, or rolling object, a vehicle, or a tool or object that slipped or swung while the worker was holding it (for example, a saw kickback or a slipping knife).
- Caught-in/between: the worker or a body part was caught in, squeezed, compressed, pinched, or crushed between objects or in running machinery, including trench, structure, or material collapses.
- Electrocution: the worker made contact with electrical energy (power lines, energized wiring or equipment).
- Other: anything else (heat, chemicals, fire or explosion, overexertion, struck against a stationary object, slips or trips without a fall).

Worked examples:
{examples}

Classify each construction incident report below into one category: Fall, Struck-by, Caught-in/between, Electrocution, or Other.
Reply with one line per report and nothing else, in this format:
N01, Fall
N02, Other

Incident reports:
{reports}"""
steps_text = """1. Copy the prompt below.
2. Open {url} (sign in with your VT account), start a new chat, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
accuracy_text = """**{version}:** {right} of {n} reports classified as OSHA classified them (**{accuracy} %**)."""
matrix_text = """Confusion matrix (rows: OSHA's classification; columns: HokieAI's output):"""
disagree_text = """Reports where HokieAI's output differs from OSHA's classification:"""
missing_text = """HokieAI's output has {got} of the {n} reports; the missing ones count as wrong."""
lab.classify_incidents(prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, accuracy_text=accuracy_text, matrix_text=matrix_text, disagree_text=disagree_text, missing_text=missing_text)


In [ ]:
#@title ▶ Step 1c · Compare Prompt Versions { display-mode: "form" }
#@markdown Run the cell after scoring the prompt versions in Step 1b. Each bar is the share of reports classified as OSHA classified them, over all outputs of that version.
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Share of correct outputs by prompt version"""
table_text = """All scored outputs so far:"""
empty_text = """No scored outputs yet: run the step above first."""
axis_text = """correct (%)"""
lab.compare("incidents", title_text=title_text, table_text=table_text, empty_text=empty_text, axis_text=axis_text)


### 📝 Report question 1
> From Step 1b: Report the accuracy of each prompt version and include the confusion matrix of the best one.

> From Step 1c: Which change to the prompt (role, definitions, worked examples) improved the accuracy most? Did repeating a version in a new chat give the same output?

> Choose two reports where HokieAI's output differs from OSHA's classification. From the report text, explain why each is hard to classify, and how you would change the prompt to fix it.

## Part 2 · Reviewing Submittals Against the Specification
Before a material is used, its submittal (here, a concrete mix design or a mortar and grout mix) is checked against the specification. Twelve practice submittals were made for this course, each to be checked against a specification excerpt adapted from the Unified Facilities Guide Specifications (UFGS 03 30 00 Cast-in-Place Concrete and 04 20 00 Masonry). A submittal states its intended use, not the portion of the structure named in the specification, so the applicable requirements must be identified first. Some values must be computed from the batch weights, some are exactly at a limit, and two submittals comply fully. The ground truth is the list of planted noncompliances. Every requirement the submittal is checked against counts as one decision (six per submittal); a flag on a requirement that sets no limit for the intended use is a false flag.

In [ ]:
#@title ▶ Step 2a · Review a Submittal with HokieAI { display-mode: "form" }
#@markdown Choose a submittal and a prompt version, run the cell, and follow the instructions. Run all three versions on at least three submittals, including a concrete and a masonry one.
#@markdown v1 · task only: the specification, the submittal, and the request to list every noncompliance.
#@markdown v2 · requirements first, then compare: asks HokieAI to list the applicable requirements first and then check them one by one (task decomposition).
#@markdown v3 · comparison table: adds a fixed comparison table and asks HokieAI to compute values the submittal does not state, such as w/cm.
#@markdown my own prompt: starts from v3; edit the prompt in the prompt box before copying it.
submittal = "Concrete mix design 1 · basement walls" #@param ["Concrete mix design 1 · basement walls", "Concrete mix design 2 · warehouse floor slab", "Concrete mix design 3 · entrance stoop and sidewalks", "Concrete mix design 4 · spread footings", "Concrete mix design 5 · stair-tower foundation walls", "Concrete mix design 6 · loading-dock apron", "Mortar and grout 1 · maintenance shop walls", "Mortar and grout 2 · office partitions", "Mortar and grout 3 · elevator pit walls", "Mortar and grout 4 · brick veneer backup walls", "Mortar and grout 5 · storage room partitions", "Mortar and grout 6 · loading-dock retaining wall"]
prompt_version = "v1 · task only" #@param ["v1 · task only", "v2 · requirements first, then compare", "v3 · comparison table", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """Review the submittal below against the specification excerpt. List every noncompliance.
{final}

Specification:
{spec}
Submittal:
{submittal}"""
v2_prompt_text = """You are a project engineer reviewing a submittal. First list every requirement in the specification excerpt that applies to the submittal's intended use. Then compare the submittal with each requirement, one by one, and decide whether it complies.
{final}

Specification:
{spec}
Submittal:
{submittal}"""
v3_prompt_text = """You are a project engineer reviewing a submittal. First list every requirement in the specification excerpt that applies to the submittal's intended use. Then compare the submittal with each requirement, one by one, in a table with the columns: requirement | required value | submitted value | complies (YES or NO). Compute any value the submittal does not state directly (for example, w/cm = water / total cementitious materials).
{final}

Specification:
{spec}
Submittal:
{submittal}"""
final_text = """End your reply with one line per noncompliance, in this format:
NONCOMPLIANT | <requirement> | <submitted value> | <required value>
If the submittal complies with every requirement, end with: NONCOMPLIANT | none"""
steps_text = """1. Copy the prompt below.
2. Open {url} (sign in with your VT account), start a new chat, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**{version}, {submittal}:** {right} of {n} requirements judged correctly; noncompliances found {found} of {planted}, false flags {false}."""
table_text = """Ground truth vs. HokieAI's output (each requirement the submittal is checked against):"""
lab.review_submittal(submittal, prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, final_text=final_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text)


In [ ]:
#@title ▶ Step 2b · Compare Prompt Versions { display-mode: "form" }
#@markdown Run the cell after Step 2a. Each bar is the share of requirements judged correctly (complies or noncompliant), over all submittals scored with that version.
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Share of correct outputs by prompt version"""
table_text = """All scored outputs so far:"""
empty_text = """No scored outputs yet: run the step above first."""
axis_text = """correct (%)"""
lab.compare("submittals", title_text=title_text, table_text=table_text, empty_text=empty_text, axis_text=axis_text)


### 📝 Report question 2
> From Step 2b: Report the share of requirements judged correctly for each prompt version, with the noncompliances found and the false flags.

> For one missed noncompliance or false flag, explain from HokieAI's output why it happened: a misread requirement, a value that had to be computed, or a requirement that does not apply to the intended use.

> Why does asking for the applicable requirements first change the result?

## Part 3 · Answering Questions from a Contract
Twenty-nine construction clauses of the Federal Acquisition Regulation (FAR), the standard terms of U.S. federal construction contracts, and 27 questions about them. Ten questions describe a project situation (a late notice, a withheld payment, a warranty claim), so the answer needs the requirement and its application to the facts. Seven cannot be answered from the text, although most have a common answer outside it. The ground truth is the answer the clauses give; quotes are checked word for word against the clauses.

In [ ]:
#@title ▶ Step 3a · Ask HokieAI About the Contract { display-mode: "form" }
#@markdown Choose a prompt version, run the cell, and follow the instructions. Score every version.
#@markdown v1 · questions only: the questions and the clauses.
#@markdown v2 · answers with quotes: asks for the sentence of the contract that supports each answer.
#@markdown v3 · only from the text, with quotes: restricts HokieAI to the clauses, asks it to apply the requirement to the facts of each situation, for the clause number and an exact quote, and for NOT IN TEXT when the clauses do not answer.
#@markdown my own prompt: starts from v3; edit the prompt in the prompt box before copying it.
prompt_version = "v1 · questions only" #@param ["v1 · questions only", "v2 · answers with quotes", "v3 · only from the text, with quotes", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """Answer the questions about the construction contract clauses below.
Reply with one line per question: Q01 | answer

Questions:
{questions}

Contract clauses:
{clauses}"""
v2_prompt_text = """Answer the questions about the construction contract clauses below. For each question, quote the sentence of the contract that supports your answer.
Reply with one line per question: Q01 | answer | exact quote

Questions:
{questions}

Contract clauses:
{clauses}"""
v3_prompt_text = """You are a contract administrator. Answer each question using only the contract clauses below; do not use outside knowledge.
If a question describes a situation, state the requirement and apply it to the facts (dates, amounts, quantities).
Reply with one line per question: Q01 | short answer | FAR clause number | exact quote of the supporting sentence, copied word for word
If the clauses do not answer the question, reply: Q01 | NOT IN TEXT | - | -

Questions:
{questions}

Contract clauses:
{clauses}"""
steps_text = """1. Copy the prompt below.
2. Open {url} (sign in with your VT account), start a new chat, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**{version}:** {right} of {n} questions answered correctly; {made_up} of the {unanswerable} questions the clauses do not answer were given an answer anyway; {verified} of {quotes} quotes found word for word in the clauses."""
table_text = """Ground truth vs. HokieAI's output:"""
lab.contract_questions(prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text)


In [ ]:
#@title ▶ Step 3b · Compare Prompt Versions { display-mode: "form" }
#@markdown Run the cell after Step 3a. Each bar is the share of questions answered correctly, including NOT IN TEXT for the questions the clauses do not answer.
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Share of correct outputs by prompt version"""
table_text = """All scored outputs so far:"""
empty_text = """No scored outputs yet: run the step above first."""
axis_text = """correct (%)"""
lab.compare("contract", title_text=title_text, table_text=table_text, empty_text=empty_text, axis_text=axis_text)


### 📝 Report question 3
> From Step 3a: Report, for each prompt version, the questions answered correctly, the answers given to questions the clauses do not answer, and the quotes found word for word.

> Choose one answer that HokieAI took from outside the clauses or misquoted. How could you detect this on a real project, where no ground truth is available?

## Part 4 · Quantity Takeoff from Drawings
Five foundation plans and five gable roofs were drawn for this course. Each comes with estimating questions (CMUs and grout; roof area, underlayment, and shingles), and the ground truth is computed exactly from the drawing's dimensions with standard estimating conventions. HokieAI reads the attached drawing and answers each question; quantities within 1–2 % of the ground truth count as correct.

In [ ]:
#@title ▶ Step 4a · Foundation Plans { display-mode: "form" }
#@markdown Choose a drawing and a prompt version, run the cell, and follow the instructions. Run all three versions on at least two drawings.
#@markdown v1 · task only: the questions and the output format.
#@markdown v2 · step-by-step reasoning: asks HokieAI to list the dimensions it reads and to solve each question step by step.
#@markdown v3 · estimating conventions: adds the estimating conventions used to compute the ground truth.
#@markdown my own prompt: starts from v3; edit the prompt in the prompt box before copying it.
drawing = "Foundation plan 1" #@param ["Foundation plan 1", "Foundation plan 2", "Foundation plan 3", "Foundation plan 4", "Foundation plan 5"]
prompt_version = "v1 · task only" #@param ["v1 · task only", "v2 · step-by-step reasoning", "v3 · estimating conventions", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """{problem}

{finals}"""
v2_prompt_text = """{problem}

First list every dimension and given value you read from the drawing and the text. Then solve each question step by step: state the formula, substitute the numbers, and compute. Check that each result is reasonable before giving it.
{finals}"""
v3_prompt_text = """{problem}

{conventions}
First list every dimension and given value you read from the drawing and the text. Then solve each question step by step: state the formula, substitute the numbers, and compute. Check that each result is reasonable before giving it.
{finals}"""
finals_text = """End your reply with one final line per question, in this format:
{lines}"""
steps_text = """1. Download {files} with the button below and copy the prompt.
2. Open {url} (sign in with your VT account), start a new chat, attach the image, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**{version}, {drawing}:** {right} of {n} quantities within tolerance of the ground truth."""
table_text = """Ground truth vs. HokieAI's output:"""
conventions_text = """Estimating conventions: total wall length = sum of the outside dimensions (no deduction at corners); wall area = length x height; one CMU covers 8 in x 16 in = 0.889 ft²; round CMUs up after adding waste. Grout = CMUs before waste x grout per CMU, converted to CY (27 ft³ = 1 CY), then add grout waste. Cells are 8 in apart, so grouting at S in on center fills 8/S of the cells."""
lab.takeoff(drawing, prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, finals_text=finals_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, conventions_text=conventions_text)


In [ ]:
#@title ▶ Step 4b · Gable Roofs { display-mode: "form" }
#@markdown Choose a drawing and a prompt version, run the cell, and follow the instructions. Run all three versions on at least two drawings.
#@markdown v1 · task only: the questions and the output format.
#@markdown v2 · step-by-step reasoning: asks HokieAI to list the dimensions it reads and to solve each question step by step.
#@markdown v3 · estimating conventions: adds the estimating conventions used to compute the ground truth.
#@markdown my own prompt: starts from v3; edit the prompt in the prompt box before copying it.
drawing = "Gable roof 1" #@param ["Gable roof 1", "Gable roof 2", "Gable roof 3", "Gable roof 4", "Gable roof 5"]
prompt_version = "v1 · task only" #@param ["v1 · task only", "v2 · step-by-step reasoning", "v3 · estimating conventions", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """{problem}

{finals}"""
v2_prompt_text = """{problem}

First list every dimension and given value you read from the drawing and the text. Then solve each question step by step: state the formula, substitute the numbers, and compute. Check that each result is reasonable before giving it.
{finals}"""
v3_prompt_text = """{problem}

{conventions}
First list every dimension and given value you read from the drawing and the text. Then solve each question step by step: state the formula, substitute the numbers, and compute. Check that each result is reasonable before giving it.
{finals}"""
finals_text = """End your reply with one final line per question, in this format:
{lines}"""
steps_text = """1. Download {files} with the button below and copy the prompt.
2. Open {url} (sign in with your VT account), start a new chat, attach the image, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**{version}, {drawing}:** {right} of {n} quantities within tolerance of the ground truth."""
table_text = """Ground truth vs. HokieAI's output:"""
conventions_text = """Estimating conventions: run = half the eave-to-eave distance; rise = run x slope; rake length = sqrt(run² + rise²); net roof area = rake length x eave length x 2 roof planes. Underlayment area = net area + ridge lap (eave length x 1 ft) + rake laps (4/12 ft x rake length x 4 rake edges); rolls = area / (roll length x width), rounded up. Starter course = eave length x exposure x 2 eaves; ridge coverage = eave length x 1 ft; rake waste = 4 rake edges x rake length x 0.25 ft²/ft; gross area = net + starter + ridge + rake waste; shingles = gross area / (3 ft x exposure), rounded up."""
lab.takeoff(drawing, prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, finals_text=finals_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, conventions_text=conventions_text)


In [ ]:
#@title ▶ Step 4c · Compare Prompt Versions { display-mode: "form" }
#@markdown Run the cell after Steps 4a and 4b. Each bar is the share of quantities within tolerance of the ground truth, over all drawings scored with that version.
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Share of correct outputs by prompt version"""
table_text = """All scored outputs so far:"""
empty_text = """No scored outputs yet: run the step above first."""
axis_text = """correct (%)"""
lab.compare("takeoff", title_text=title_text, table_text=table_text, empty_text=empty_text, axis_text=axis_text)


### 📝 Report question 4
> From Step 4c: Report the share of correct quantities for each prompt version, across at least two foundation plans and two roofs.

> For one wrong quantity, find in HokieAI's output where the error came from: a misread dimension, a different estimating convention, or an arithmetic error.

> Why does stating the estimating conventions help more than asking for step-by-step reasoning?

## Part 5 · Reading Schedules on a Drawing Sheet
Five 36 × 24 in drawing sheets, made for this course, each carry the door schedules of several unit types, a window schedule, hardware sets, and door-type elevations. The task: list every door of one unit type. The ground truth is the schedule's own data; a door counts as correct only if all six of its fields are transcribed exactly.

In [ ]:
#@title ▶ Step 5a · Door Schedules { display-mode: "form" }
#@markdown Choose a sheet and a prompt version, run the cell, and follow the instructions. Run all three versions on at least two sheets.
#@markdown v1 · whole sheet: HokieAI receives the whole sheet.
#@markdown v2 · cropped schedule: HokieAI receives only the part of the schedule for the unit type.
#@markdown v3 · cropped schedule and column names: adds the column names and asks for an exact transcription.
#@markdown my own prompt: starts from v3 with both images; edit the prompt in the prompt box before copying it.
sheet = "Door schedule sheet 1 (unit type 1A)" #@param ["Door schedule sheet 1 (unit type 1A)", "Door schedule sheet 2 (unit type 1.1)", "Door schedule sheet 3 (unit type 2A)", "Door schedule sheet 4 (unit type 1.2)", "Door schedule sheet 5 (unit type 1A)"]
prompt_version = "v1 · whole sheet" #@param ["v1 · whole sheet", "v2 · cropped schedule", "v3 · cropped schedule and column names", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """The attached drawing sheet contains a door schedule. List every door of UNIT TYPE {unit}.
{format}"""
v2_prompt_text = """The attached image is the part of a door schedule for UNIT TYPE {unit}. List every door.
{format}"""
v3_prompt_text = """The attached image is the part of a door schedule for UNIT TYPE {unit}. The schedule's columns are, left to right: MARK, DOOR TYPE, DESCRIPTION, WIDTH, HEIGHT, FRAME, FIRE RATING, HDWR SET, LOCATION. Transcribe every row exactly as written; if a value is not legible, write ?.
{format}"""
format_text = """Reply with one line per door and nothing else, in this format:
mark | door type | width | height | hardware set | location
1 | DF | 3'-0" | 7'-0" | 32 | ENTRY"""
steps_text = """1. Download {files} with the button below and copy the prompt.
2. Open {url} (sign in with your VT account), start a new chat, attach the image, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**{version}, {sheet}:** {right} of {n} doors transcribed exactly ({fields_right} of {fields} fields right)."""
table_text = """Ground truth vs. HokieAI's output:"""
lab.schedule(sheet, prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, format_text=format_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text)


In [ ]:
#@title ▶ Step 5b · Compare Prompt Versions { display-mode: "form" }
#@markdown Run the cell after Step 5a. Each bar is the share of doors transcribed exactly, over all sheets scored with that version.
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Share of correct outputs by prompt version"""
table_text = """All scored outputs so far:"""
empty_text = """No scored outputs yet: run the step above first."""
axis_text = """correct (%)"""
lab.compare("schedule", title_text=title_text, table_text=table_text, empty_text=empty_text, axis_text=axis_text)


### 📝 Report question 5
> From Step 5b: Report the number of doors transcribed exactly with the whole sheet and with the cropped schedule. Why does cropping change the result?

> Before using a door count from HokieAI in an estimate, what would you check, and how?

### Data and model sources
- Incident reports: Severe Injury Reports (2015–2025), Occupational Safety and Health Administration, U.S. Department of Labor, public domain, https://www.osha.gov/severe-injury-reports.
- Specification excerpts: adapted from the Unified Facilities Guide Specifications 03 30 00 and 04 20 00, U.S. Department of Defense, public domain, https://www.wbdg.org/dod/ufgs; the submittals are practice submittals made for CEM4644.
- Contract clauses: Federal Acquisition Regulation, Part 52 (29 construction clauses), public domain, https://www.acquisition.gov/far.
- Drawings: original practice drawings made for CEM4644 (not for construction).
- Generative AI: GPT models via HokieAI (Virginia Tech).